In [1]:
import sqlite3, sys, pandas as pd, json
from pathlib import Path


In [2]:
# Connect to our project root
project_root = Path.cwd()

if not (project_root / "src").is_dir():
    project_root = project_root.parent

sys.path.insert(0, str(project_root))

from src.load_cloudtrail import load_cloudtrail
from src.cloudtrail_adapter import adapt_cloudtrail_event

print("Project root:", project_root)


Project root: /Users/gerardoramirez/Library/CloudStorage/GoogleDrive-gerardoram1010@gmail.com/My Drive/Portfolio Projects/cloud-security-intelligence-platform


In [3]:
db_path = project_root / "data" / "cloudtrail.db"

connection = sqlite3.connect(db_path)

In [4]:
file_path = project_root / "data" / "synthetic_cloudtrail.json"

raw_events = load_cloudtrail(file_path)

normalized_events = [
    adapt_cloudtrail_event(record)
    for record in raw_events
]

df = pd.DataFrame(normalized_events)

Invalid source IP address.
Invalid source IP address.
Invalid source IP address.
Invalid source IP address.
Invalid source IP address.
Invalid source IP address.
Invalid source IP address.
Invalid source IP address.
Invalid source IP address.
Invalid source IP address.


In [5]:
# Convert timestamps to UTC datetime
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)

# Using JSON seralization to create json strings of the record that can be checked for duplicates
df["raw_record_json"] = df["raw_record"].apply(
    lambda record: json.dumps(record, sort_keys=True)
)

#Remove duplicate rows
df_clean = df.drop_duplicates(
    subset=["raw_record_json"]
).copy()

#Creaet event date using timestamp 
df_clean["event_date"] = df_clean["timestamp"].dt.date

# Calculating failure rate

df_clean["failed"] = df_clean["raw_record"].apply(
    lambda record: "errorCode" in record
)

In [6]:
df_clean.head()

,timestamp,user,action,region,resource,security_event,ip_metadata,raw_record,raw_record_json,event_date,failed
0,2026-11-11 07:00:00+00:00,admin,ConsoleLogin,eu-west-1,analytics-logs,SecurityEvent(timestamp='2026-11-11T07:00:00+0...,"{'source_ip_raw': '198.51.100.5', 'source_ip_t...","{'eventTime': '2026-11-11T07:00:00+00:00', 'ev...","{""awsRegion"": ""eu-west-1"", ""errorCode"": ""Acces...",2026-11-11,True
1,2026-11-10 02:00:00+00:00,admin,PutObject,us-west-2,analytics-logs,SecurityEvent(timestamp='2026-11-10T02:00:00+0...,"{'source_ip_raw': '203.0.113.42', 'source_ip_t...","{'eventTime': '2026-11-10T02:00:00+00:00', 'ev...","{""awsRegion"": ""us-west-2"", ""eventName"": ""PutOb...",2026-11-10,False
2,2026-11-14 13:00:00+00:00,admin,PutObject,us-east-1,analytics-logs,SecurityEvent(timestamp='2026-11-14T13:00:00+0...,"{'source_ip_raw': '192.0.2.15', 'source_ip_typ...","{'eventTime': '2026-11-14T13:00:00+00:00', 'ev...","{""awsRegion"": ""us-east-1"", ""eventName"": ""PutOb...",2026-11-14,False
3,2026-11-12 08:00:00+00:00,admin,ConsoleLogin,eu-west-1,analytics-logs,SecurityEvent(timestamp='2026-11-12T08:00:00+0...,"{'source_ip_raw': '198.51.100.5', 'source_ip_t...","{'eventTime': '2026-11-12T08:00:00+00:00', 'ev...","{""awsRegion"": ""eu-west-1"", ""eventName"": ""Conso...",2026-11-12,False
4,2026-11-10 11:00:00+00:00,developer,ConsoleLogin,us-east-1,analytics-logs,SecurityEvent(timestamp='2026-11-10T11:00:00+0...,"{'source_ip_raw': '198.51.100.5', 'source_ip_t...","{'eventTime': '2026-11-10T11:00:00+00:00', 'ev...","{""awsRegion"": ""us-east-1"", ""eventName"": ""Conso...",2026-11-10,False


In [7]:
df_sql = df_clean[
    ["timestamp", "user", "action", "region", "failed"]
].copy()

In [8]:
print(type(df_sql))
print(df_sql.shape)
print(df_sql.head())

<class 'pandas.DataFrame'>
(95, 5)
                  timestamp       user        action     region  failed
0 2026-11-11 07:00:00+00:00      admin  ConsoleLogin  eu-west-1    True
1 2026-11-10 02:00:00+00:00      admin     PutObject  us-west-2   False
2 2026-11-14 13:00:00+00:00      admin     PutObject  us-east-1   False
3 2026-11-12 08:00:00+00:00      admin  ConsoleLogin  eu-west-1   False
4 2026-11-10 11:00:00+00:00  developer  ConsoleLogin  us-east-1   False


In [9]:
# Convert failed column to integers 0, 1
df_sql["failed"] = df_sql["failed"].astype(int)

# Convert timestamp to standardized ISO 8601 strings
df_sql["timestamp"] = df_sql["timestamp"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")

In [10]:
# Check our new datatypes
df_sql.dtypes

timestamp      str
user           str
action         str
region         str
failed       int64
dtype: object

In [11]:
print(df_sql.dtypes)
print(df_sql.shape)
print(df_sql.head())

timestamp      str
user           str
action         str
region         str
failed       int64
dtype: object
(95, 5)
              timestamp       user        action     region  failed
0  2026-11-11T07:00:00Z      admin  ConsoleLogin  eu-west-1       1
1  2026-11-10T02:00:00Z      admin     PutObject  us-west-2       0
2  2026-11-14T13:00:00Z      admin     PutObject  us-east-1       0
3  2026-11-12T08:00:00Z      admin  ConsoleLogin  eu-west-1       0
4  2026-11-10T11:00:00Z  developer  ConsoleLogin  us-east-1       0


In [12]:
print(df_sql["failed"].value_counts())

failed
0    86
1     9
Name: count, dtype: int64


In [13]:
df_sql.to_sql(
    name="events",
    con=connection,
    if_exists="replace",
    index=False
)

95

In [14]:
pd.read_sql_query(
    "SELECT COUNT(*) AS total_events FROM events",
    connection
)

,total_events
0,95


In [15]:
pd.read_sql_query(
    """SELECT COUNT(*) AS failed_events
    FROM events 
    WHERE failed == 1""",
    connection
)

,failed_events
0,9


In [16]:
pd.read_sql_query(
    """SELECT user, COUNT(*) AS failed_events
    FROM events 
    WHERE failed = 1
    GROUP BY user""",
    connection
)

,user,failed_events
0,admin,4
1,developer,2
2,service-user,3


In [17]:
pd.read_sql_query(
    """SELECT 
        user,
        COUNT(*) AS event_count,
        SUM(failed) AS failed_events
    FROM events
    GROUP BY user""",
    connection
)

,user,event_count,failed_events
0,admin,36,4
1,analyst,18,0
2,developer,22,2
3,service-user,19,3


In [18]:
pd.read_sql_query(
    """SELECT 
        user,
        COUNT(*) AS event_count,
        SUM(failed) AS failed_events,
        SUM(failed) * 1.0 / COUNT(*) * 100 AS failure_rate
    FROM events
    GROUP BY user""",
    connection
)

,user,event_count,failed_events,failure_rate
0,admin,36,4,11.111111
1,analyst,18,0,0.000000
2,developer,22,2,9.090909
3,service-user,19,3,15.789474


In [19]:
pd.read_sql_query(
    """SELECT 
        user,
        COUNT(*) AS event_count,
        SUM(failed) AS failed_events,
        SUM(failed) * 1.0 / COUNT(*) * 100 AS failure_rate
    FROM events
    GROUP BY user
    HAVING event_count > 20""",
    connection
)

,user,event_count,failed_events,failure_rate
0,admin,36,4,11.111111
1,developer,22,2,9.090909


In [20]:
pd.read_sql_query(
    """SELECT 
        user,
        COUNT(*) AS event_count,
        SUM(failed) AS failed_events,
        SUM(failed) * 1.0 / COUNT(*) * 100 AS failure_rate
    FROM events
    GROUP BY user
    HAVING failure_rate > 10.0""",
    connection
)

,user,event_count,failed_events,failure_rate
0,admin,36,4,11.111111
1,service-user,19,3,15.789474


In [21]:
df_sql_result = pd.read_sql_query(
    """SELECT 
        user,
        COUNT(*) AS event_count,
        SUM(failed) AS failed_events,
        SUM(failed) * 1.0 / COUNT(*) * 100 AS failure_rate
    FROM events
    GROUP BY user""",
    connection
)

assert df_sql_result["event_count"].sum() == len(df_clean)
assert df_sql_result["failed_events"].sum() == df_clean["failed"].sum()

### Day 17 — SQLite Integration and SQL Aggregations

Loaded 95 deduplicated synthetic CloudTrail events into a local SQLite database using pandas and Python's `sqlite3` library. Converted timestamps into UTC-formatted strings and Boolean failure indicators into integers for SQL compatibility.

**Key accomplishments:**
- Created an `events` table containing individual CloudTrail event records.
- Practiced `SELECT`, `WHERE`, `GROUP BY`, and `HAVING` queries.
- Calculated event counts, failure counts, and failure rates by user.
- Corrected integer division to preserve accurate percentage calculations.
- Validated SQL aggregation totals against the cleaned pandas DataFrame using assertions.

**Results:** 95 events, 9 failures, and an overall failure rate of 9.47%.

**Key takeaway:** Preserving event-level data allows flexible SQL aggregations and further investigation. SQL queries must account for filtering, data types, and duplicate records to produce reliable analytical results.

**Limitation:** All records are synthetic, and failed events do not necessarily indicate malicious activity.